# ANOVA: Plant Growth

Each method is its own editable block. Contrasts and weights must follow the printed group order.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PLOTS_DIR = ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

def find_data_file(file_name):
    candidates = [ROOT / file_name, ROOT / "data" / file_name]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {file_name}. Put it in the project root or data/ folder and include the extension."
    )

def save_plot(fig, file_name):
    path = PLOTS_DIR / file_name
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved plot: {path}")
    return path

from anova_toolbox import ANOVAToolbox

use_default_dataset = True
custom_file_name = "my_anova_data.csv"

data_path = ROOT / "data" / "plant_growth.csv" if use_default_dataset else find_data_file(custom_file_name)
tool = ANOVAToolbox(data_path)
df = tool.data.copy()

group_col = "group"
value_col = "weight"
if group_col not in df.columns or value_col not in df.columns:
    raise ValueError(f"Set group_col and value_col to columns in your data. Available columns: {list(df.columns)}")
if pd.to_numeric(df[value_col], errors="coerce").dropna().empty:
    raise ValueError(f"{value_col} must contain numeric values.")
if df[group_col].dropna().nunique() < 2:
    raise ValueError(f"{group_col} must contain at least two groups.")

print("Dataset:", data_path)
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("missing_values"))
display(tool.summarize_data(group_col, value_col))
print("Group order:", list(tool.group_summary(group_col, value_col).index))

In [ ]:
# Editable block: one-way ANOVA
group_col = "group"
value_col = "weight"
alpha = 0.05

result = tool.one_way_anova(group_col, value_col, alpha, label="One-way ANOVA")
display(tool.anova_table(result))
print(result["decision"])
fig, ax = tool.plot_group_boxplot(group_col, value_col)
save_plot(fig, "anova_group_boxplot.png")
fig, ax = tool.plot_group_means_ci(group_col, value_col)
save_plot(fig, "anova_group_means_ci.png")
fig, ax = tool.plot_f_distribution(result)
save_plot(fig, "anova_one_way_f_critical_region.png")
fig, ax = tool.plot_residual_diagnostics(result)
save_plot(fig, "anova_residuals_vs_fitted.png")

In [ ]:
# Editable block: two-way ANOVA without replication
# The default PlantGrowth data have one treatment factor. This block creates a block factor by observation order.
row_factor = "block"
col_factor = "group"
value_col = "weight"
alpha = 0.05

two_way_df = df.copy()
if row_factor not in two_way_df.columns:
    two_way_df[row_factor] = two_way_df.groupby(col_factor).cumcount() + 1
two_way_tool = ANOVAToolbox(two_way_df)
result = two_way_tool.two_way_anova_no_replication(row_factor, col_factor, value_col, alpha, label="Two-way ANOVA without replication")
display(result["anova_table"])

In [ ]:
# Editable block: ANOVA contrast test
group_col = "group"
value_col = "weight"
group_order = list(tool.group_summary(group_col, value_col).index)
print("Group order:", group_order)
contrast_vector = [-1, 0, 1]
null_value = 0
alternative = "two-sided"
alpha = 0.05

result = tool.contrast_test(group_col, value_col, contrast_vector, null_value, alternative, alpha, label="ANOVA contrast test")
display(pd.DataFrame([result]))
fig, ax = tool.plot_t_distribution_for_contrast(result)
save_plot(fig, "anova_contrast_t_critical_region.png")

In [ ]:
# Editable block: linear combination confidence interval
group_col = "group"
value_col = "weight"
group_order = list(tool.group_summary(group_col, value_col).index)
print("Group order:", group_order)
weights = [0.5, 0.5, 0]
confidence = 0.95

result = tool.linear_combination_ci(group_col, value_col, weights, confidence, label="ANOVA linear combination CI")
display(pd.DataFrame([result]))

In [ ]:
# Editable block: Bonferroni pairwise comparisons
group_col = "group"
value_col = "weight"
alpha = 0.05

display(tool.bonferroni_pairwise(group_col, value_col, alpha))